# Inventario
# ETL Pipeline - Network Inventory

## Setup

This notebook reads input files from `../data/input/` and generates output in `../data/output/`

**Required input files:**
- `SEG_MT_YYYYMM_MOD_ENTREGA_sample.csv`
- `SEG_MT_YYYYMM_UT_TODAS_sample.csv`
- `MPROTEC_sample.xlsx`
- `Jerarquía_sample.xlsx`

**Output:**
- `inventory_sample.parquet` in `../data/output/`

In [1]:
import os
import pandas as pd
import datetime
import numpy as np

# Directorio en donde se encuentran los archivos
directorio = '../data/input'
output_dir = '../data/output'

## Funciones

In [66]:
## Veo por cada empalme el detalle del cable anterior y siguiente en la secuencia (SECO o API) y defino el tipo de empalme (SECO-SECO, TRANSICION, API-API, Sin secuencia)

def tipo_de_empalme(row):
    if row['TIPO_OBJ_TEC'] != 'EMPALME MT':
        return 'NO'
    mm = row['TPLMA']
    secuencia_actual = row['SECUENCIA']

    # Filtramos el DataFrame 'cables' por el mismo TPLMA una sola vez para optimizar
    mismo_tplma = cables[cables['TPLMA'] == mm]

    # Buscamos el registro con la secuencia más cercana hacia atrás
    anterior_df = mismo_tplma[mismo_tplma['SECUENCIA'] < secuencia_actual]
    # Buscamos el registro con la secuencia más cercana hacia adelante
    posterior_df = mismo_tplma[mismo_tplma['SECUENCIA'] > secuencia_actual]

    if anterior_df.empty or posterior_df.empty:
        return 'Sin secuencia'

    # .idxmax() e .idxmin() nos dan el índice del valor más cercano
    idx_anterior = anterior_df['SECUENCIA'].idxmax()
    idx_posterior = posterior_df['SECUENCIA'].idxmin()

    anterior_val = anterior_df.loc[idx_anterior, 'detalle']
    posterior_val = posterior_df.loc[idx_posterior, 'detalle']

    # Validamos nulos
    if pd.isna(anterior_val) or pd.isna(posterior_val):
        return 'Sin secuencia'

    # Lógica de clasificación
    if anterior_val == 'SECO' and posterior_val == 'SECO':
        return 'SECO-SECO'
    
    if anterior_val == 'API' and posterior_val == 'API':
        return 'API-API'

    if (anterior_val == 'API' and posterior_val == 'SECO') or \
       (anterior_val == 'SECO' and posterior_val == 'API'):
        return 'TRANSICION'

    return 'OTRO'

## Leo Archivos

In [ ]:
# Mes del archivo #
mes = datetime.datetime.now().month - 1
anio = datetime.datetime.now().year
if mes <= 0:
    mes = 12
    anio = anio - 1

print(mes, anio)

if mes <= 9:
    mes = '0' + str(mes)

print(mes, anio)

# Archivo Módulos de Entrega  #
print('Leyendo Módulos de Entrega')
Entrega = pd.read_csv(f'{directorio}/SEG_MT_{anio}{mes}_MOD_ENTREGA_sample.csv', sep='\t', encoding='latin1')
Entrega = Entrega.drop_duplicates(subset=['UT_OBJECTNAMEID'])

# UTs de MT #
print('Leyendo UTs MT')
uts = pd.read_csv(f'{directorio}/SEG_MT_{anio}{mes}_UT_TODAS_sample.csv', sep='\t', encoding='latin1')

# Módulos de Protección #
print('Leyendo Módulos de Protección')
mprotec = pd.read_excel(f'{directorio}/MPROTEC_sample.xlsx')

# Jerarquía - SE
print('Leyendo Jerarquía')
alimentadores  = pd.read_excel(directorio + '/Jerarquia_sample.xlsx')
alimentadores = alimentadores[['SE','Emplaz']]
alimentadores.drop_duplicates(inplace=True)
alimentadores = alimentadores.rename(columns={'Emplaz' : 'EMPLAZAMIENTO','SE':'NOMBRE'})
alimentadores = alimentadores[~alimentadores['EMPLAZAMIENTO'].str.contains('SE', na=False)]
alimentadores = alimentadores[~alimentadores['EMPLAZAMIENTO'].isna()]

3 2026
03 2026
Leyendo Módulos de Entrega
Leyendo UTs MT
Leyendo Módulos de Protección
Leyendo Jerarquía


## Editando UTs

In [68]:
uts2 = uts.copy()

In [69]:
uts2.head()

,#TPLNR,DESCRIPT,TIPO_OBJ_TEC,INVNR,BEGRU,F_ALTA_SAP,F_BAJA_SAP,PTA_SERVICIO,TPLMA,EMPLAZAMIENTO,...,COOR_X,COOR_Y,CTRO_COSTO,CALLE,ALTURA_DESDE,ALTURA_HASTA,ENTRE_CALLE_1,ENTRE_CALLE_2,GPS,IDENTIFICADOR
0,AK983,PIQUETE AK983,PIQUETE_MT,5579339,MT,22/03/2016,NaN,31/10/2022,MM102269053,13212A,...,5596314.693,6146008.986,4424.0,CNO A MORENO*,15701,15799,NaN,NaN,"-34.8275 , -58.9472",AK983
1,AF329-004,PIQUETE AF329,PIQUETE_MT,94103736,MT,14/04/2024,NaN,01/01/1998,MM96025328,15326A,...,5630845.780,6199138.375,4439.0,ARROYO TORO,NaN,NaN,ARROYO TORO,NaN,"-34.3448 , -58.578",AF329
2,AF238-004,PIQUETE AF238,PIQUETE_MT,94087427,MT,14/04/2024,NaN,01/01/1995,MM6326281,15326A,...,5630618.331,6194585.130,4439.0,ARROYO CURUBICA,NaN,NaN,RIO SARMIENTO,NaN,"-34.3859 , -58.5797",AF238
3,AG294-004,PIQUETE AG294,PIQUETE_MT,94178230,MT,22/03/2016,NaN,01/01/1998,MM95858046,15326A,...,5627541.249,6206856.433,4439.0,RIO PARANA DE LAS PALMAS,NaN,NaN,RIO PARANA DE LAS PALMAS,NaN,"-34.2757 , -58.615",AG294
4,AL287,PIQUETE AL287,PIQUETE_MT,5582662,MT,22/03/2016,NaN,01/01/1985,MM5582643,13212A,...,5597669.836,6148663.899,4424.0,NaN,NaN,NaN,NaN,NaN,"-34.8034 , -58.9327",AL287


### Detalle

In [70]:
uts2 = uts.copy()

# Completo longitud de ALIM con 0
uts2['LONGITUD'] = np.where(uts2['TIPO_OBJ_TEC'] == 'ALIM', 0, uts['LONGITUD'])

# Creo campo "detalle" en base a DESCRIPT con tipo de cable y tipo de línea ---------------------------------------------------

print('Colocando detalle cable y línea')
uts2['detalle'] = np.where(
    uts2['DESCRIPT'].str.endswith(' API'), 'API', #es api?

    np.where(
        uts2['DESCRIPT'].str.contains('XLPE'), 'SECO', # es seco?

            np.where(
                uts2['DESCRIPT'].str.startswith('VANO '), # es VANO

                # De acá todos los casos con "VANO"
                np.where(
                    uts2['DESCRIPT'].str.endswith('T'), 'TRIANGULAR',

                    np.where(
                        uts2['DESCRIPT'].str.endswith('P. H'), 'HORIZONTAL PROT',

                        np.where(
                            uts2['DESCRIPT'].str.endswith('H'), 'HORIZONTAL DESN',

                            np.where(
                                uts2['DESCRIPT'].str.endswith('P. V'), 'VERTICAL PROT',

                                np.where(
                                    uts2['DESCRIPT'].str.endswith('V'), 'VERTICAL DESN',

                                    np.where(
                                        uts2['DESCRIPT'].str.endswith('P. LP'), 'LINE POST PROT',

                                        np.where(
                                            uts2['DESCRIPT'].str.endswith('LP'),
                                            'LINE POST DESN', 'COMPACTA'
                ))))))),
                #---------------------------------------------------------------------------------------------


                np.where(
                    uts2['DESCRIPT'].str.contains('TRAMO'),
                    np.where(uts2['DESCRIPT'].str.endswith('Al-I'), 'SECO', None), None))))

# Detalle en Piquetes --------------------------------------------------------------------------------------
# Asignar 'Madera' a 'detalle' para filas donde 'OBJETO' comienza con 'PO' o 'POD'
print('Colocando detalle postes (Hormigon, Madera, Metálico)')
uts2['detalle'] = np.where((uts2['OBJETO'].str.startswith('PO') |
                            uts2['OBJETO'].str.startswith('POD')) &
                            (uts2['OBJETO'].notna()), 'Madera', uts2['detalle'])

# Asignar 'Hormigon' a 'detalle' para filas donde 'OBJETO' comienza con 'PH'
uts2['detalle'] = np.where(uts2['OBJETO'].str.startswith('PH') &
                           (uts2['OBJETO'].notna()), 'Hormigon', uts2['detalle'])

# Asignar 'Metalico' a 'detalle' para filas donde 'OBJETO' comienza con 'PHME'
uts2['detalle'] = np.where(uts2['OBJETO'].str.startswith('PHME') &
                           (uts2['OBJETO'].notna()), 'Metalico', uts2['detalle'])

# Completamos el detalle en CENTROS con el tipo de CENTRO --------------------------------------------------------
print('Colocando detalle de centros')
lista = ['CCE', 'CCN',  'CCS', 'CCTN', 'CCTS',  'CTA2', 'CTAB', 'CTAMH',
'CTAMM', 'CTAS', 'CTE', 'CTI', 'CTN',  'CTS', 'CTSG']
otros = ['CCPE', 'CP', 'CTPE', 'CTPI', 'DM_CS', 'MCS3', 'MCS4', 'MCU3']
uts2['TIPO_OBJ_TEC'] = uts2['TIPO_OBJ_TEC'].str.replace('_', ' ')


for i in lista:      # Tipos de centros más comunes
    uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                               (uts2['ALIAS'] == i) &
                               (uts2['OBJETO'].notna()), i, uts2['detalle'])

for o in otros:     # Agrupamos como OTROS a los tipos menos comunes
    uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                               (uts2['ALIAS'] == o) &
                               (uts2['OBJETO'].notna()), 'OTRO', uts2['detalle'])

uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &         # OTROS tmb si es null
                           (uts2['OBJETO'].isnull()), 'OTRO', uts2['detalle'])

# Detalle de seccionadores ----------------------------------------------------------------
print('Colocando detalle de seccionadores')
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'REC') & (uts2['OBJETO'].notna()),
                           'R', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECA') & (uts2['OBJETO'].notna()),
                           'K', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECBC') & (uts2['OBJETO'].notna()),
                           'BC', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECT') & (uts2['OBJETO'].notna()),
                           'S', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECU') & (uts2['OBJETO'].notna()),
                           'S', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECZ') & (uts2['OBJETO'].notna()),
                           'Z', uts2['detalle'])
uts2['detalle'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                           (uts2['ALIAS'] == 'SECZE') & (uts2['OBJETO'].notna()),
                           'Z', uts2['detalle'])



Colocando detalle cable y línea
Colocando detalle postes (Hormigon, Madera, Metálico)
Colocando detalle de centros
Colocando detalle de seccionadores


### TIPO_OBJ_TEC y Tipo_Cable

In [71]:
# Corrigiendo TIPO_OBJ_TEC -----------------------------------------------------------------------
# Cámara
print('Corrigiendo tipos de objeto tecnico')
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                                (uts2['ALIAS'].isnull()), 'CAMARA', uts2['TIPO_OBJ_TEC'])

# Tipo Seccionador
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "SECCIONADR") &
                                (uts2['ALIAS'].str.startswith('SE') |
                                 uts2['ALIAS'].str.startswith('REC')) &
                                (uts2['OBJETO'].notna()), 'SECCIONADOR', uts2['TIPO_OBJ_TEC'])


# Tipo Camara
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "CENTRO") &
                                ((uts2['ALIAS'].str.startswith('C')) |
                                (uts2['ALIAS'].str.startswith('MC')) |
                                (uts2['ALIAS'].str.startswith('DM'))
                                ) &
                                (uts2['OBJETO'].notna()),
                                'CAMARA', uts2['TIPO_OBJ_TEC'])


# Tipo Plataforma
uts2['TIPO_OBJ_TEC'] = np.where((uts2['TIPO_OBJ_TEC'] == "CAMARA") &
                                (uts2['ALIAS'].str.startswith('CTA')) &
                                (uts2['OBJETO'].notna()), 'PLATAFORMA', uts2['TIPO_OBJ_TEC'])


# Aux Inv 2--------------------------------
print('Calculando tipo_cable')
uts2['tipo_cable'] = np.where(uts2['DESCRIPT'].str.contains('TRAMO '), uts2['DESCRIPT'].str.replace('TRAMO ', ''),
                           np.where(uts2['DESCRIPT'].str.contains('VANO '), uts2['DESCRIPT'].str.replace('VANO ', ''), None))

Corrigiendo tipos de objeto tecnico
Calculando tipo_cable


## Agregando MPROT

In [72]:
# Colocación MPROT
print('Asignando MPROTS')

mprotecuts = mprotec.copy()
indices_to_drop = mprotec[(mprotec['PERUT'].str.startswith('MM')) &
                          (mprotec['PERUT'].notna())].index
mprotecuts = mprotec.drop(indices_to_drop)

uts2 = pd.merge(uts2, mprotecuts, how='left',  left_on='#TPLNR', right_on='PERUT')

mprotecmm = mprotec[(mprotec['PERUT'].str.startswith('MM')) & (mprotec['PERUT'].notna())]
uts2 = pd.merge(uts2, mprotecmm, how='left',  left_on='TPLMA', right_on='PERUT')

uts2['MPROT'] = uts2['MPROT_x'].combine_first(uts2['MPROT_y'])
uts2['NIVEL PROT'] = uts2['NIVEL PROT_x'].combine_first(uts2['NIVEL PROT_y'])
print(uts2[(uts2['NIVEL PROT'].notna()) & (uts2['NIVEL PROT']>0)]['NIVEL PROT'].unique())
uts2['NIVEL PROT'] = uts2['NIVEL PROT'].fillna(0)
uts2['NIVEL PROT'] = uts2['NIVEL PROT'].astype(int)
print(uts2[(uts2['NIVEL PROT'].notna()) & (uts2['NIVEL PROT']>0)]['NIVEL PROT'].unique())
print(uts2['NIVEL PROT'].isnull().sum())

# Nivel Red
print('Colocando nivel red')
uts2['NIVEL_RED'] = np.where(uts2['MPROT'].str.startswith('MP_R'), 'Troncal',
                        np.where(uts2['MPROT'].str.startswith('MP_K'), 'Apendice K',
                            np.where(uts2['MPROT'].str.startswith('MP_Z'), 'Apendice Z', 'Troncal'
                            )
                        )
                    )

# MPROT de Alimentador ----------------------------------------------------------------
uts2['MPROT'] = uts2['MPROT'].astype(str)

uts2['MPROT'] = np.where(
    (uts2['MPROT'].str.endswith('A')) &
    (~uts2['MPROT'].str.contains('Z')) &
    (~uts2['MPROT'].str.contains('R')) &
    (~uts2['MPROT'].str.contains('K')),
    'MP_' + uts2['EMPLAZAMIENTO'],
    uts2['MPROT']
)

Asignando MPROTS
[4. 6. 7. 5. 8. 2. 3. 1.]
[4 6 7 5 8 2 3 1]
0
Colocando nivel red


## Otros campos

In [73]:
# Colocación Secuencia -----------------------------------------------------------------------------------
print('Colocando Secuencia')
uts3 = uts2.copy()

uts3 = pd.merge(uts3, Entrega[['UT_OBJECTNAMEID', 'ME_NOMBRE', 'ME_ORDEN', 'SECUENCIA', 'ME_PSEUDO']],
                how='left',  left_on='OBJECTNAMEID', right_on='UT_OBJECTNAMEID')

uts3['TPLMA'] = np.where((uts3['TPLMA'].str.startswith('MM')) & (uts3['TPLMA'] != uts3['ME_PSEUDO']), uts3['ME_PSEUDO'], uts3['TPLMA'])

uts3 = uts3.drop(['ME_PSEUDO'], axis=1)

uts3['ME_ORDEN'] = uts3['ME_ORDEN'].fillna(0).astype(int)
uts3['SECUENCIA'] = uts3['SECUENCIA'].fillna(0).astype(int)

# Colocación detalle de celdas -------------------------------------------------------------------------------
print('Detalle de celdas')
uts3[['ELEMENTO',
      'TIPO ELEMENTO',
      'DESTINO',
      'TIPO DESTINO']] = uts3['IDENTIFICADOR'].str.split('-', expand=True)

mapeo_tipo_elemento = {
    'CLMI': 'Medición Cliente MT',
    'FAEK': 'Seccionador Autodesc Fusible',
    'IMIMT': 'Interruptor',
    'RFMT': 'Ruptofusible',
    'SBCT': 'Seccionador',
    'SUMB': 'Seccionador'
}

mapeo_tipo_destino = {
    0: '',
    '0': '',
    'CLMI': 'Cliente MT',
    'ECTA': 'Alim, Centro o EEMM',
    'FMIMT': 'Cliente MT',
    'TMMB': 'Transformador'
}
# Reemplazar los códigos en las columnas
uts3['TIPO ELEMENTO'] = uts3['TIPO ELEMENTO'].fillna(0).replace(mapeo_tipo_elemento).astype(str)
uts3['TIPO DESTINO'] = uts3['TIPO DESTINO'].fillna(0).replace(mapeo_tipo_destino).astype(str)


# Separación Lat y Long ---------------------------------------------------------------------
print('Separando Latitud y Longitud')
uts3['Lat'] = uts3['GPS'].str.split(',').str.get(0)
uts3['Long'] = uts3['GPS'].str.split(',').str.get(1)

# Eliminando columnas -----------------------------------------------------------------------
print('Eliminando columnas innescesarias')
uts3 = uts3.drop(['PERUT_x', 'MPROT_x',
                  'NIVEL PROT_x', 'PERUT_y',
                  'MPROT_y', 'NIVEL PROT_y',
                  'COOR_X', 'COOR_Y', 'BEGRU',
                  'INVNR', 'DIVISION', 'OBJECTID',
                  'PARTIDO_GEO', 'LOCALIDAD_GEO',
                  'FLTYP', 'CTRO_COSTO', 'UT_OBJECTNAMEID',
                  'GPS','PERFIL','PARTE_OBJETO','SECCION'], axis=1)

# Cambiando casos particulares de EMPLAZAMIENTO ----------------------------------------------------
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6211-A', '6211A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6211-B', '6211B')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6216-A', '6216A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('6216-B', '6216B')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('16986 - FI', '16986A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].replace('16976 - FI', '16976A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].fillna('00A')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('_', '')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('-', '')

# Nivel de tensión a los alimentadores de barra 7 y 8 -----------------------------------------------
uts3['tipo_cable'] = np.where(uts3['TIPO_OBJ_TEC'] == 'ALIM',
                        np.where(
                            uts3['EMPLAZAMIENTO'].str[-3:].str[0:2].astype(int) > 70, '33', '13'),
                               uts3['tipo_cable'])

## Colocar nombre de CT a celdas --------------------------------------------------------------------
print('Colocando nombre de CT a celdas')
sec_mt = uts3[uts3['TIPO'] == 'SECC_MTCT'][['#TPLNR', 'TPLMA']]
sec_mt.rename(columns={'#TPLNR': 'trafo', 'TPLMA': 'celda'}, inplace=True)
uts3= pd.merge(uts3, sec_mt, how='left',  left_on='#TPLNR', right_on='celda')
uts3['tipo_cable'] = np.where(uts3['TIPO'] == 'CELDA', uts3['trafo'], uts3['tipo_cable'])


## Borrando columnas sobrantes --------------------------------------------------------------------
print('Borrando columnas sobrantes')
uts3 = uts3.drop(['trafo'], axis=1)
uts3 = uts3.drop(['celda'], axis=1)
uts3 = uts3.drop(['ALIAS'], axis=1)
uts3 = uts3.drop(['SPRID'], axis=1)

## Colocar coordenadas a ICC -----------------------------------------------------------
print('Colocando coordenadas a ICC')
ubis = uts3[uts3['TIPO_OBJ_TEC'] == 'CELDA MT'][['#TPLNR', 'Lat', 'Long']]
ubis.rename(columns={'#TPLNR': 'ubi', 'Lat': 'lati', 'Long': 'longi'}, inplace=True)
iccs = uts3[uts3['TIPO_OBJ_TEC'] == 'ICC'][['#TPLNR', 'TPLMA']]
iccs = pd.merge(iccs, ubis, how='left',  left_on='TPLMA', right_on='ubi')
iccs = iccs.drop(['ubi'], axis=1)
iccs.rename(columns={'#TPLNR': 'icc', 'TPLMA': 'secc'}, inplace=True)
uts3 = pd.merge(uts3, iccs, how='left',  left_on='#TPLNR', right_on='icc')
uts3['Lat'] = np.where(uts3['TIPO_OBJ_TEC'] == 'ICC', uts3['lati'], uts3['Lat'])
uts3['Long'] = np.where(uts3['TIPO_OBJ_TEC'] == 'ICC', uts3['longi'], uts3['Long'])
uts3 = uts3.drop(['lati'], axis=1)
uts3 = uts3.drop(['longi'], axis=1)
uts3 = uts3.drop(['secc'], axis=1)
uts3 = uts3.drop(['icc'], axis=1)

# Saco FIN de EMPLAZAMIENTO y ALIM con FIN en DESCRIPT -------------------------------------
uts3 = uts3[~((uts3['DESCRIPT'].str.contains('FIN')) & (uts3['TIPO_OBJ_TEC'] == 'ALIM'))]
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('FIN ', '')
uts3['EMPLAZAMIENTO'] = uts3['EMPLAZAMIENTO'].str.replace('FIN', '')


# Agrego Subestación --------------------------------------------------------------------------
uts3 = uts3.merge(alimentadores[['EMPLAZAMIENTO', 'NOMBRE']], how='left',  left_on='EMPLAZAMIENTO', right_on='EMPLAZAMIENTO')
uts3.rename(columns={'NOMBRE': 'SUBESTACION'}, inplace=True)

# Corrigo caso particular de PTO_SERVICIO -----------------------------------------------------------------
uts3['PTA_SERVICIO'] = np.where((uts3['PTA_SERVICIO'] == '23/10/5025') & (uts3['PTA_SERVICIO'].notna()), '23/10/2025', uts3['PTA_SERVICIO'])

Colocando Secuencia
Detalle de celdas
Separando Latitud y Longitud
Eliminando columnas innescesarias
Colocando nombre de CT a celdas
Borrando columnas sobrantes
Colocando coordenadas a ICC


## Cálculo de tipo de empalme

In [74]:
# Armo un df con cables y empalmes con secuencia
cables = uts3[
    (uts3['TIPO_OBJ_TEC'] == 'CABLE MT') 
    ][['#TPLNR', 'TIPO_OBJ_TEC', 'TPLMA', 'SECUENCIA', 'detalle']].reset_index(drop=True).copy()

empalmes = uts3[
    (uts3['TIPO_OBJ_TEC'] == 'EMPALME MT')
    ][['#TPLNR', 'TIPO_OBJ_TEC', 'TPLMA', 'SECUENCIA', 'detalle']].reset_index(drop=True).copy()

cable_empalme = pd.concat([cables, empalmes], axis=0).sort_values(['TPLMA', 'SECUENCIA']).reset_index()

In [75]:
# Aplico la función para calcular el tipo de empalme
cable_empalme['tipo_de_empalme'] = cable_empalme.apply(tipo_de_empalme, axis=1)

In [76]:
# Completo los na y agrego el tipo de empalme en df uts3
cable_empalme['tipo_de_empalme'] = cable_empalme['tipo_de_empalme'].fillna('NO')
uts3 = uts3.merge(cable_empalme[['#TPLNR', 'tipo_de_empalme']], how='left', on='#TPLNR').copy()
uts3['detalle'] = uts3['detalle'].combine_first(uts3['tipo_de_empalme'])
uts3.drop('tipo_de_empalme', axis=1, inplace=True)

## Guardo el inventario final en parquet

In [ ]:
uts3.to_parquet(output_dir + '/Inventario_MT_sample.parquet', index=False)